# AMD-Latent
### Split of the dataset

---

The current notebook splits the dataset into training, validation and test set for AMD score classification tasks

# Packages import and functions defining

Loading the required packages and definition of functions.

In [ ]:
# PACKAGES IMPORT

# System packages
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Jupyter packages
from amdlatent.utils.widgets import filechooser, slideroptions, toggleoptions

# Dataset packages
from amdlatent.dataset.audit import naneval
from amdlatent.dataset.extraction import filteringimages, datasetsplit

# Path checker
if Path().resolve().name != "AMD-Latent": os.chdir(os.path.join(Path().resolve().parents[1]))

# Code execution

Select the following options for running the script and execute it.

In [ ]:
# OPTIONS SELECTION

dirselect = filechooser("Select the <b>AMD Study</b> folder:")
splitselect = toggleoptions(["Couples-wise", "Patient-wise", "Couples stratified-wise"],"Select the <b>dataset split type</b>:")
constrslider = slideroptions("Select the <b>construction set percentage</b> for the dataset split:", 0, 1, .1, .8)
trainslider = slideroptions("Select the <b>training set percentage</b> for the construction set split:", 0, 1, .1, .8)

In [ ]:
# SCRIPT EXECUTION

metadata = {
    "directory": dirselect.selected_path,
    "constr": constrslider.value if constrslider is not None else None,
    "train": trainslider.value if trainslider is not None else None,
    "splittype": splitselect.value if splitselect is not None else None,
}

dataset = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='AMD')
dataset.columns = dataset.columns.astype(str).str.strip()
enrollment = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='enrollment_randomization')
enrollment.columns = enrollment.columns.astype(str).str.strip()
followup = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='followup')
followup.columns = followup.columns.astype(str).str.strip()

paths = np.load(os.path.join("results", "areds", "dataset", "encoding", "retclip", "imagepaths.npy"))
paths = pd.DataFrame(paths)
paths = paths.apply(lambda col: col.map(lambda p: os.path.join(metadata["directory"], "Data", "EDO", *Path(p).parts[-5:])))
blacklist = naneval(os.path.join(metadata["directory"], "Data", "EDO"), dataset)
filtpaths = filteringimages(paths, blacklist)
if metadata["splittype"] == "Couples stratified-wise": trainpaths, valpaths, testpaths = datasetsplit(filtpaths, splittype=metadata["splittype"], constrsize=metadata["constr"], trainsize=metadata["train"], dataset=dataset)
else: trainpaths, valpaths, testpaths = datasetsplit(filtpaths, splittype=metadata["splittype"], constrsize=metadata["constr"], trainsize=metadata["train"])
if not os.path.exists(os.path.join("results", "areds", "dataset", "splits")):
    os.makedirs(os.path.join("results", "areds", "dataset", "splits"))
trainpaths.to_csv(os.path.join("results", "areds", "dataset", "splits",  f"trainpaths_{metadata['splittype'].split('-')[0].lower().replace(' ', '')}.csv"), index=False)
valpaths.to_csv(os.path.join("results", "areds", "dataset", "splits", f"valpaths_{metadata['splittype'].split('-')[0].lower().replace(' ', '')}.csv"), index=False)
testpaths.to_csv(os.path.join("results", "areds", "dataset", "splits", f"testpaths_{metadata['splittype'].split('-')[0].lower().replace(' ', '')}.csv"), index=False)